In [0]:
%sql
-- ============================================
-- CRIAÇÃO DA TABELA DELTA
-- Tabela: bandeira_acionada
-- Camada: RAW
-- ============================================

CREATE TABLE IF NOT EXISTS mba.raw.bandeira_acionada
(
    DatGeracaoConjuntoDados DATE
        COMMENT 'Data do processamento de carga automática no momento da geração para publicação do conjunto de dados abertos.',

    DatCompetencia DATE
        COMMENT 'Data de competência do acionamento do registro.',

    NomBandeiraAcionada STRING
        COMMENT 'Nome da bandeira acionada.',

    VlrAdicionalBandeira DECIMAL(20,2)
        COMMENT 'Valor do acionamento da bandeira.'
)
USING DELTA
COMMENT 'Tabela contendo informações sobre o acionamento de bandeiras tarifárias.';

In [0]:
# IMPORA AS BIBLIOTECAS
from pyspark.sql.functions import (
    col,
    trim,
    to_date,
    regexp_replace,
    when
)

# CONFIGURAÇÃO DOS CAMINHOS
volume_path = "/Volumes/mba/stage/dados_bruto/ANEEL"

In [0]:
# ============================================
# LEITURA DO ARQUIVO
# BANDEIRA TARIFÁRIA - ACIONAMENTO
# ============================================

df_bandeira_acionada = (
    spark.read
    .option("header", "true")
    .option("sep", ";")
    .option("encoding", "UTF-8")
    .csv(f"{volume_path}/ANEEL-bandeira-tarifaria-acionamento.csv")
)

# ============================================
# TRANSFORMAÇÃO DOS TIPOS DE DADOS
# ============================================

df_bandeira_acionada = (
    df_bandeira_acionada
    .withColumn(
        "DatGeracaoConjuntoDados",
        to_date(col("DatGeracaoConjuntoDados"), "yyyy-MM-dd")
    )
    .withColumn(
        "DatCompetencia",
        to_date(col("DatCompetencia"), "yyyy-MM-dd")
    )
    .withColumn(
        "VlrAdicionalBandeira",
        regexp_replace(
            regexp_replace(col("VlrAdicionalBandeira"), "\\.", ""),
            ",",
            "."
        ).cast("decimal(20,2)")
    )
)

# ============================================
# GRAVAÇÃO NA DELTA TABLE
# ============================================

(
    df_bandeira_acionada.write
    .mode("overwrite")
    .saveAsTable("mba.raw.bandeira_acionada")
)

In [0]:
dbutils.notebook.exit("success")

In [0]:
%sql
select * from mba.raw.bandeira_acionada

In [0]:
%sql

SELECT
    date_format(DatCompetencia, 'yyyyMM') AS MesCompetencia,
    case when NomBandeiraAcionada like 'Vermelha%' then 1 else 0 end AS IsVermelha
FROM mba.raw.bandeira_acionada;